# Inference — final CatBoost pathogenicity classifier

Loads the **stage-2** deployable CatBoost model
(`models/catboost_final_pathogenicity.cbm` + `*_meta.json` from
`notebooks/modeling/final/train_final_catboost.ipynb` / `python src/final_catboost.py`)
and scores **real ClinVar VUS** (uncertain significance) that were cleaned, joined
to UniProt, and position-matched separately from the pathogenic/benign training set.

Honest KPIs live in nested CV (`notebooks/modeling/cv/cv_catboost.ipynb`), not here.
XGBoost inference is under `final/deprecated/`.

Build the VUS table first (once):

```bash
python src/clean_clinvar.py
python src/join_clinvar_uniprot.py --vus
python src/position_matching_clinvar_uniprot.py --vus
```


In [1]:
from pathlib import Path
import os
import sys

import pandas as pd
from dotenv import load_dotenv


In [2]:
load_dotenv()
project_root = Path(os.environ["PROJECT_ROOT"])
sys.path.insert(0, str(project_root / "src"))

from final_catboost import (
    DEFAULT_META_PATH,
    DEFAULT_MODEL_PATH,
    load_final_catboost,
    predict_pathogenicity,
)
from features import (
    POSITION_MATCHED_VUS_IN,
    prepare_inference_frame,
)

INPUT_PARQUET = POSITION_MATCHED_VUS_IN
OUT_PARQUET = project_root / "data/processed/vus_inference_predictions.parquet"


## Load model


In [3]:
model, meta = load_final_catboost(DEFAULT_MODEL_PATH, DEFAULT_META_PATH)
print(f"n_estimators={meta['n_estimators']}, threshold={meta['threshold']:.4f}")
print(f"trained on {meta['n_train']} variants / {meta['n_genes']} genes")
print("features:", meta["feature_columns"])
pd.Series(meta["params"], name="param").to_frame()


n_estimators=132, threshold=0.4950
trained on 11563 variants / 137 genes
features: ['protein_position', 'relative_protein_position', 'distance_to_closest_feature', 'has_protein_position', 'in_domain', 'in_region', 'in_zinc_finger', 'in_active_site', 'in_binding_site', 'in_disulfide', 'in_mod_res', 'in_functional_site', 'in_any_feature', 'closest_feature_type', 'ReferenceAlleleVCF', 'AlternateAlleleVCF']


,param
subsample,0.70
learning_rate,0.05
l2_leaf_reg,1.00
depth,6.00


## Score ClinVar VUS

These variants were **never used as training labels**. Predictions are the model's
pathogenic-vs-benign call at the saved Youden threshold (not a ClinVar reclassification).

`y_proba` is a **ranking score**, not a calibrated P(pathogenic). Nested-CV OOF
Brier ≈ 0.21 / ECE ≈ 0.04; `auto_class_weights="Balanced"` pulls scores toward 0.5,
and gene-pool base rates (36–74% pathogenic) are not in the score. Do not read 0.92
as “92% likely pathogenic.” Rank VUS with `y_proba`; use `pred_label` at the Youden cutoff.


In [4]:
raw = pd.read_parquet(INPUT_PARQUET)
frame = prepare_inference_frame(raw, labels={"vus"})
print(f"VUS to score: {len(frame):,} variants / {frame['gene'].nunique():,} genes")
print(frame["gene"].value_counts().head(10))

preds = predict_pathogenicity(frame, model=model, meta=meta)
print("\nPredicted labels:")
print(preds["pred_label"].value_counts())
print(f"\ny_proba: median={preds['y_proba'].median():.3f}, "
      f"mean={preds['y_proba'].mean():.3f}")
preds.sort_values("y_proba", ascending=False).head(15)


VUS to score: 3,506 variants / 123 genes
gene
RUNX1     768
LDLR      300
RYR1      218
PAH       163
HNF1A     151
MYOC      150
HNF4A     100
DICER1     86
GCK        85
ITGA2B     84
Name: count, dtype: int64

Predicted labels:
pred_label
pathogenic    2076
benign        1430
Name: count, dtype: int64

y_proba: median=0.601, mean=0.546


,VariationID,gene,Name,ClinicalSignificance,label,y_proba,threshold,y_pred,pred_label
3201,3068511,GCK,NM_000162.5(GCK):c.883G>C (p.Gly295Arg),Uncertain significance,vus,0.921965,0.495034,1,pathogenic
316,120290,PAH,NM_000277.3(PAH):c.869A>T (p.His290Leu),Uncertain significance,vus,0.915905,0.495034,1,pathogenic
3236,3242388,F9,NM_000133.4(F9):c.1303T>A (p.Cys435Ser),Uncertain significance,vus,0.909216,0.495034,1,pathogenic
220,90844,MSH2,NM_000251.3(MSH2):c.2005G>C (p.Gly669Arg),Uncertain significance,vus,0.908316,0.495034,1,pathogenic
2631,1693233,PAH,NM_000277.3(PAH):c.667A>T (p.Asn223Tyr),Uncertain significance,vus,0.898736,0.495034,1,pathogenic
2306,1327562,PAH,NM_000277.3(PAH):c.1204T>A (p.Phe402Ile),Uncertain significance,vus,0.893051,0.495034,1,pathogenic
3112,2757681,RUNX1,NM_001754.5(RUNX1):c.346T>A (p.Phe116Ile),Uncertain significance,vus,0.890886,0.495034,1,pathogenic
2589,1684419,RUNX1,NM_001754.5(RUNX1):c.335T>A (p.Leu112Gln),Uncertain significance,vus,0.890313,0.495034,1,pathogenic
262,102535,PAH,NM_000277.3(PAH):c.1132A>T (p.Thr378Ser),Uncertain significance,vus,0.890052,0.495034,1,pathogenic
3047,2674648,ITGA2B,NM_000419.5(ITGA2B):c.286T>A (p.Cys96Ser),Uncertain significance,vus,0.889818,0.495034,1,pathogenic


In [5]:
OUT_PARQUET.parent.mkdir(parents=True, exist_ok=True)
preds.to_parquet(OUT_PARQUET, index=False)
print("Saved", OUT_PARQUET, preds.shape)


Saved /home/milijanas/projects/clinvar-pathogenicity-classifier/data/processed/vus_inference_predictions.parquet (3506, 9)
